# 🚚⚠️📊 Прогнозирование риска задержки доставки CargaPronto: сегментация клиентов и бинарная классификация заказов

Автор: Якунин Михаил

Дата: "__" _______ 2026 г.

***По этой ссылке можно посмотреть проект и скачать последнюю версию: https://github.com/MEYakunin/Sprint18_Delivery_Delay_Prediction.git***

## Цель проекта

*Цель проекта* — построить для логистической компании CargaPronto модель бинарной классификации, которая по данным о заказе и профиле клиента будет предсказывать риск задержки доставки (`late_delivery_risk`) ещё в момент оформления заказа. 

*Конечный продукт* — система поддержки принятия решений для диспетчеров: при высоком риске опоздания (>70%) система должна выдавать предупреждение и рекомендовать сменить приоритет доставки, что позволит компании активно управлять ситуацией вместо того, чтобы констатировать факт нарушения SLA.

*Ключевая задача исследования* — проверить гипотезу транспортного департамента о «неоднородности» клиентской базы: подтвердить, что риск задержки определяется не только параметрами самого заказа, но и географическим положением клиента, а также его поведенческим профилем (частота заказов, сумма покупок, доля возвратов, чувствительность к скидкам). Для этого клиенты будут сегментированы с помощью кластеризации K-Means++ на логистические зоны (`geo_cluster`) и поведенческие сегменты (`beh_cluster`), после чего полученные признаки добавляются в обучающие данные моделей.

*Целевой показатель качества* — ROC-AUC > 0.75 на тестовой выборке: метрика оценивает способность модели ранжировать заказы по степени риска, что позволит диспетчерам точечно выделять доставки, требующие повышенного внимания.

## Задача проекта

*Задача проекта* — разработать модель бинарной классификации, которая в момент оформления заказа предсказывает целевую переменную `late_delivery_risk`: 1 — заказ доставлен с задержкой, 0 — доставлен вовремя. Входными данными служат параметры самого заказа (`order_id`, `customer_id`, `order_date`, `shipping_mode`, `category_name`, `item_price`) и профиль клиента из справочника (`customer_lat`, `customer_lon`, `recency`, `total_orders`, `total_sales`, `return_rate` и `avg_discount`).

*Инструменты моделирования* — логистическая регрессия (быстрая и прозрачная, задаёт «нижнюю планку» качества) и `CatBoostClassifier` (градиентный бустинг, способный находить сложные закономерности). Метрика качества — `ROC-AUC`: она оценивает способность модели ранжировать заказы по степени риска, что позволяет диспетчерам точечно выделять доставки, требующие повышенного внимания. Удовлетворительный результат — `ROC-AUC` > 0.75 на тестовой выборке.

*План работы:*
1. Загрузка датасетов `ds_s18_customers.csv` и `ds_s18_orders.csv`, технический аудит, устранение полных дубликатов, проверка целостности данных, подготовка временных признаков `order_month`, `order_weekday`, `order_hour`.
2. EDA: проверка баланса классов, географический аудит и удаление аномалий координат, анализ зависимостей признаков.
3. Разделение заказов на обучающую, валидационную и тестовую выборки в соотношении 60:20:20 с помощью `GroupShuffleSplit` (клиенты не должны пересекаться между выборками).
4. Обучение базовых моделей (логистическая регрессия и `CatBoostClassifier`) на признаках заказа — фиксация точки отсчёта качества.
5. Кластеризация клиентов: геокластеры (`geo_cluster`) по координатам и поведенческие сегменты (`beh_cluster`) по RFM-признакам, с выбором числа кластеров методом локтя и защитой от утечки данных.
6. Обучение финальных моделей на расширенном наборе признаков и оценка ROC-AUC на валидационной выборке.
7. Тестирование лучшей модели на тестовой выборке: итоговый ROC-AUC, матрица ошибок, анализ важности признаков.
8. Выводы и бизнес-рекомендации для CargaPronto.

## Описание данных

В проекте используются два датасета, выгруженные из ERP-системы компании CargaPronto.

1. `ds_s18_customers.csv` — профили уникальных клиентов

Справочник содержит агрегированную информацию о каждом клиенте: географическое положение и поведенческие показатели, собранные по методу RFM.

| Признак | Описание |
| :--- | :--- |
| `customer_id` | Уникальный номер клиента в системе ERP; ключ для связи с таблицей заказов |
| `customer_lat` | Широта местонахождения клиента |
| `customer_lon` | Долгота местонахождения клиента |
| `recency` | Количество дней, прошедших с момента последнего заказа до точки отсчёта (Recency) |
| `total_orders` | Общее количество совершённых заказов (Frequency) |
| `total_sales` | Суммарная выручка, которую клиент принёс компании за всё время (Monetary) |
| `avg_discount` | Средний размер скидки, которую получает клиент; маркер чувствительности к акциям |
| `return_rate` | Доля проблемных заказов (возвраты и отмены); ключевой показатель нагрузки на курьеров |

2. `ds_s18_orders.csv` — заказы клиентов

Таблица событий, где зафиксирован каждый факт продажи и итог доставки; здесь находится целевая переменная.

| Признак | Описание |
| :--- | :--- |
| `order_id` | Уникальный номер заказа |
| `customer_id` | Идентификатор клиента; ключ для связи с профилем из справочника |
| `order_date` | Дата и время совершения покупки |
| `category_name` | Категория заказанного товара |
| `item_price` | Стоимость товара в заказе |
| `shipping_mode` | Режим доставки: Standard Class, First Class, Second Class, Same Day |
| `late_delivery_risk` | Целевая переменная: 1 — заказ доставлен с задержкой; 0 — заказ доставлен вовремя |

## Содержание <a id="content"></a>

* [1. Импорт библиотек, выполнение базовых настроек](#1)
* [2. Загрузка данных, первичное знакомство](#2)
  * [2.1. Технический аудит данных](#2_1)
  * [2.2. Устранение дубликатов и проверка целостности (ID-Check)](#2_2)
  * [2.3. Подготовка признаков даты и времени](#2_3)
  * [2.4. Промежуточный итог](#2_4)
* [3. EDA](#3)
  * [3.1. Проверка баланса классов](#3_1)
  * [3.2. Географический аудит](#3_2)
  * [3.3. Удаление аномалий](#3_3)
  * [3.4. Анализ зависимостей между признаками](#3_4)
* [4. Разделение на выборки](#4)
* [5. Обучение базовой модели](#5)
  * [5.1. Логистическая регрессия](#5_1)
  * [5.2. CatBoostClassifier](#5_2)
  * [5.3. Оценка качества на валидационной выборке](#5_3)
* [6. Кластеризация по признакам, связанным с местоположением](#6)
  * [6.1. Подготовка признаков и защита от утечки данных](#6_1)
  * [6.2. Выбор числа кластеров методом локтя](#6_2)
  * [6.3. Визуализация кластеров и центроидов](#6_3)
* [7. Кластеризация по признакам профилей клиентов](#7)
  * [7.1. Подготовка векторов признаков и предобработка](#7_1)
  * [7.2. Выбор числа кластеров методом локтя](#7_2)
  * [7.3. Анализ полученных сегментов](#7_3)
  * [7.4. Визуализация кластеров с помощью t-SNE](#7_4)
* [8. Обучение модели на новых признаках](#8)
  * [8.1. Добавление `geo_cluster` и `beh_cluster` в данные](#8_1)
  * [8.2. Обучение финальных моделей](#8_2)
  * [8.3. Оценка ROC-AUC на валидационной выборке](#8_3)
* [9. Дополнительное задание — подбор лучшего количества кластеров](#9)
* [10. Тестирование лучшей модели](#10)
  * [10.1. Предсказание на тестовой выборке и итоговый ROC-AUC](#10_1)
  * [10.2. Матрица ошибок](#10_2)
  * [10.3. Анализ важности признаков](#10_3)
* [11. Выводы и рекомендации](#11)

## [1. Импорт библиотек, выполнение базовых настроек](#content)<a id="1"></a>

Загрузите все необходимые для выполнения проекта библиотеки и другие компоненты.



In [ ]:
#%pip install -r requirements.txt

In [ ]:
# Стандартные библиотеки

# Сторонние библиотеки
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
import phik

from catboost import CatBoostClassifier

from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    confusion_matrix, 
    ConfusionMatrixDisplay
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)

In [ ]:
# Устанавка константных параметров
RANDOM_STATE = 42
TEST_SIZE = 0.2

# Рабочие границы: долгота -125…-20; широта > 0 (северное полушарие)
LON_MIN, LON_MAX = -125, -20
LAT_MIN = 0

In [ ]:
# Настройки Pandas для отображения DataFrame
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}'.replace(',', ' '))
pd.set_option('display.memory_usage', True)

# Настройка стиля графиков
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("viridis")

# Отключает научную нотацию (экспоненциальную запись) на оси Y для всех графиков
mpl.rcParams['axes.formatter.useoffset'] = False
mpl.rcParams['axes.formatter.limits'] = [-99, 99]

# Подавляем INFO-сообщения Optuna, оставляем только предупреждения и ошибки 
optuna.logging.set_verbosity(optuna.logging.WARNING)

## [2. Загрузка данных, первичное знакомство](#content)<a id="2"></a>

* Загрузите датасеты `ds_s18_customers.csv` и `ds_s18_orders.csv`.
  * Путь к первому файлу — `'/datasets/ds_s18_customers.csv'`.
  * Путь ко второму файлу — `'/datasets/ds_s18_orders.csv'`.
* Проведите технический аудит данных.
* Выявите и устраните полные дубликаты строк.
* Выполните проверку целостности данных (ID-Check).
* Подготовьте признаки, связанные с датой и временем.
* Создайте дополнительные колонки `order_month`, `order_weekday`, `order_hour`.

### [2.1. Технический аудит данных](#content)<a id="2_1"></a>

In [ ]:
# Загружаем датасеты
df_customers = pd.read_csv('datasets/ds_s18_customers.csv')
df_orders = pd.read_csv('datasets/ds_s18_orders.csv')

# Создание словаря c датафреймами
dataframes = {
    'customers': df_customers,
    'orders': df_orders
}

In [ ]:
# Создаем функцию для отображения данных
def primary_info(df, name):
    """
    Выводит базовую сводную информацию о DataFrame: заголовок с именем таблицы,
    первые 5 строк, сводку типов и пропусков, а также описательную статистику.
    """
    print('='*50)
    print(f'Информация о таблице: {name}')
    print('='*50)
    display(df.head(5))
    df.info()
    display(df.describe(include='all').T)
    print()

In [ ]:
# Выводим первичную информацию 
for name, df in dataframes.items():
    primary_info(df, name)

#### Промежуточный итог

1. Обе таблицы загружены без ошибок: `df_customers` — 20 652 строки × 8 колонок, `df_orders` — 180 519 строк × 7 колонок. Полных пропусков нет, что упрощает дальнейшую предобработку.

2. Типы данных в целом корректны, однако колонка `order_date` в `df_orders` хранится как строка (`str`) — на этапе подготовки признаков её необходимо преобразовать в тип `datetime`.

3. Категориальные признаки информативны: 4 режима доставки (доминирует Standard Class) и 50 категорий товаров (лидер — Cleats).

4. В географических координатах `customer_lon` обнаружены аномалии (пределы диапазона долготы –125 до –20): минимальное -158.026° и максимальное 115.263° — точки вне региона работы компании (США и Пуэрто-Рико). Эти выбросы будут удалены на этапе EDA, так как K-Means чувствителен к ним.

5. Память таблиц невелика (1.3 MB и 9.6 MB), оптимизация dtype не требуется.

*Вывод:* данные пригодны для дальнейшей работы. Следующие шаги — проверка полных дубликатов, контроль целостности связей (ID-Check) и подготовка временных признаков `order_month`, `order_weekday`, `order_hour`.

### [2.2. Устранение дубликатов и проверка целостности (ID-Check)](#content)<a id="2_2"></a>

In [ ]:
# Проверяем полные дубликаты
print(f'Полных дубликатов в customers: {df_customers.duplicated().sum()}')
print(f'Полных дубликатов в orders:    {df_orders.duplicated().sum()}')

In [ ]:
# Проверяем строки с повторяющимся customer_id и order_id
cust_dup = df_customers[df_customers.duplicated(subset=['customer_id'], keep=False)]
ord_dup = df_orders[df_orders.duplicated(subset=['order_id'], keep=False)]

print(f'Строк с повторяющимся customer_id в df_customers: {len(cust_dup)}')
print(f'Строк с повторяющимся order_id в df_orders: {len(ord_dup)}')

In [ ]:
# Проверяем строки с повторяющимся признаками кроме order_id
subset_cols = df_orders.columns.drop('order_id')
print(f'Почти-дубли в orders: {df_orders.duplicated(subset=subset_cols).sum()}')

In [ ]:
# Выводим строки с повторяющимся признаками кроме order_id
mask = df_orders.duplicated(subset=subset_cols, keep=False)
df_orders[mask].sort_values(['customer_id', 'order_date', 'item_price']).head(22)

In [ ]:
# Удаляем строки с повторяющимся признаками кроме order_id
df_orders.drop_duplicates(subset=subset_cols, keep=False, inplace=True)
print(f'Почти-дубли в orders: {df_orders.duplicated(subset=subset_cols).sum()}')

In [ ]:
# Определяем строки с заказами без профиля клиента
orphan_mask = ~df_orders['customer_id'].isin(df_customers['customer_id'])
print('Заказов без профиля клиента:', orphan_mask.sum())

# Определяем профили клиентов без заказа
no_orders_mask = ~df_customers['customer_id'].isin(df_orders['customer_id'])
print('Клиентов без заказов:', no_orders_mask.sum())

In [ ]:
# Удаляем клиентов без заказа
df_customers = df_customers[~no_orders_mask]
no_orders_mask = ~df_customers['customer_id'].isin(df_orders['customer_id'])
print('Клиентов без заказов:', no_orders_mask.sum())

In [ ]:
print(f'Итоговый размер датасета df_orders: {df_orders.shape}')
print(f'Итоговый размер датасета df_customers: {df_customers.shape}')

#### Промежуточный итог

1. *Полных дубликатов нет:* в `df_customers` — 0, в `df_orders` — 0. Первичные ключи обеих таблиц уникальны — повторяющихся `customer_id` и `order_id` не обнаружено.

2. *«Почти-дубли» устранены:* 20 806 строк полностью совпадали по всем атрибутам заказа (категория товара, режим доставки, цена, дата, результат доставки) и различались только номером `order_id` — это следы технического сбоя при выгрузке из ERP. Все строки, участвующие в дублировании, удалены (`keep=False`); после чистки дублей не осталось.

3. *ID-Check пройден:* заказов без профиля клиента — 0, целостность связи `orders → customers` подтверждена. При этом в справочнике найдено 16 профилей без единого заказа — такие записи непригодны ни для кластеризации, ни для слияния, поэтому они удалены из `df_customers`.

4. *Итоговые размеры таблиц:* `df_orders` — 140 633 строки × 7 колонок (удалено 39 886 строк, ≈ 22.1% исходных данных), `df_customers` — 20 636 строк × 8 колонок (удалено 16 профилей).

*Вывод:* подраздел выполнен в соответствии с ТЗ: дубликаты и «почти-дубли» устранены, целостность связей между таблицами подтверждена, справочник очищен от неиспользуемых профилей.

### [2.3. Подготовка признаков даты и времени](#content)<a id="2_3"></a>

In [ ]:
# Приводим признак с датой к формату datetime
df_orders['order_date'] = pd.to_datetime(df_orders['order_date'])
df_orders.info()

In [ ]:
# Выделяем новые признаки из даты
df_orders['order_year'] = df_orders['order_date'].dt.year
df_orders['order_month'] = df_orders['order_date'].dt.month
df_orders['order_weekday'] = df_orders['order_date'].dt.weekday
df_orders['order_hour'] = df_orders['order_date'].dt.hour

# Выделяем циклические признаки для месяца
df_orders['order_month_sin'] = np.sin(2 * np.pi * df_orders['order_month'] / 12)
df_orders['order_month_cos'] = np.cos(2 * np.pi * df_orders['order_month'] / 12)

In [ ]:
# Удаляем исходный признак
df_orders.drop(['order_date'], axis=1, inplace=True)
df_orders.head()

#### Промежуточный итог

1. *Тип данных исправлен:* колонка `order_date` приведена к формату `datetime64` с помощью `pd.to_datetime`; пропусков после преобразования нет.

2. *Созданы временные признаки:* `order_year` (год), `order_month` (месяц), `order_weekday` (день недели, пн = 0) и `order_hour` (час оформления заказа) — получены через аксессор `.dt` из исходной даты.

3. *Добавлены циклические признаки для месяца* (`order_month_sin`, `order_month_cos`): они сохраняют «близость» декабря и января, которую числовая кодировка месяцев теряет, — это поможет моделям корректно учитывать сезонность.

4. *Целостность данных сохранена:* таблица содержит 140 633 строки, новых пропусков не появилось;

*Вывод:* дата преобразована в `datetime`, созданы обязательные признаки `order_month`, `order_weekday`, `order_hour` и дополнительные временные фичи.

### [2.4. Промежуточный итог](#content)<a id="2_4"></a>

1. *Данные загружены и проверены.* Исходные размеры: `df_customers` — 20 652 строки × 8 колонок, `df_orders` — 180 519 строк × 7 колонок. Пропусков нет; единственная некорректная колонка — `order_date` (строка), приведена к типу `datetime` в подразделе 2.3.

2. *Дубликаты устранены.* Полных дубликатов не обнаружено (0 в обеих таблицах), однако найдено 20 806 «почти-дублей» — строк, полностью совпадающих по всем атрибутам заказа и различающихся только `order_id` (признак повторной выгрузки из ERP). Все участвующие строки удалены; первичные ключи `customer_id` и `order_id` уникальны.

3. *Целостность связей подтверждена (ID-Check):* заказов без профиля клиента — 0, каждый заказ имеет справочную запись. При этом 16 профилей клиентов не имели ни одного заказа и были удалены как непригодные для кластеризации и слияния.

4. *Итоговые размеры таблиц:* `df_orders` — 140 633 строки × 7 колонок (удалено 39 886 строк, ≈ 22.1%), `df_customers` — 20 636 строк × 8 колонок. Баланс целевой переменной после чистки ≈ 54.9% — смещения данных не произошло.

7. *Временные признаки подготовлены:* `order_year`, `order_month`, `order_weekday`, `order_hour` (обязательные по ТЗ) и дополнительные циклические `order_month_sin`, `order_month_cos` для учёта сезонности; исходная `order_date` заменена производными признаками.

## [3. EDA](#content)<a id="3"></a>

На этом этапе ваша цель — исследовать структуру данных и подтвердить гипотезы транспортного департамента о «неоднородности» базы. Кроме того, вам предстоит изучить и удалить географические аномалии.

1. Проверьте баланс классов.
2. Выполните географический аудит.
3. Удалите аномалии.
4. Проанализируйте зависимости между признаками.



### [3.1. Проверка баланса классов](#content)<a id="3_1"></a>

In [ ]:
# Рассчитываем количество и доли целевой переменной
class_counts = df_orders['late_delivery_risk'].value_counts().sort_index()
class_share = df_orders['late_delivery_risk'].value_counts(normalize=True).sort_index()

labels = ['0 — вовремя', '1 — с задержкой']
colors = ['#4C72B0', '#DD8452']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Строим столбчатую диаграмму количества
bars = axes[0].bar(labels, class_counts.values, color=colors, edgecolor='black')
for bar, share in zip(bars, class_counts.values):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1500,
        f'{share}',
        ha='center', fontsize=12, fontweight='bold'
    )
axes[0].set_title('Целевая переменная late_delivery_risk\n(абсолютные частоты)')
axes[0].set_ylabel('Количество заказов')
axes[0].set_ylim(0, class_counts.max() * 1.15)

# Строим круговую диаграмму долей
axes[1].pie(
    class_share.values,
    labels=labels,
    autopct='%1.1f%%',
    startangle=90,
    colors=colors,
    explode=(0, 0.05),
    textprops={'fontsize': 11}
)
axes[1].set_title('Баланс классов (доли)')

plt.tight_layout()
plt.show()

#### Промежуточный итог

1. Доля заказов с задержкой (`late_delivery_risk = 1`) составляет **≈ 54.9%**, вовремя — **≈ 45.1%**. Это подтверждает цифру после внутреннего аудита (55% задержек) и гипотезу о критической ситуации с соблюдением SLA.

2. Дисбаланс классов умеренный (разница ~10 п.п.), выраженного перекоса нет, поэтому специальные техники борьбы с дисбалансом (взвешивание классов, `oversampling`/`undersampling`) на этапе моделирования не требуются.

3. Выбранная метрика `ROC-AUC` устойчива к такому смещению и корректно оценивает способность модели ранжировать заказы по риску, что соответствует бизнес-задаче `CargaPronto`.

### [3.2. Географический аудит](#content)<a id="3_2"></a>

In [ ]:
# Создаем функцию для отображения графика с регионами
def plot_regions(df):
    """
    Строит диаграмму рассеяния координат клиентов и выделяет аномалии.
    """
    fig, ax = plt.subplots(figsize=(12, 8))

    # Основное облако точек и аномалии разными цветами
    for status, color in zip(['В регионе (США/Пуэрто-Рико)', 'Аномалия («цифровой шум»)'], ["#27C033", "#9352DD"]):
        subset = df[df['geo_status'] == status]
        ax.scatter(subset['customer_lon'], subset['customer_lat'], alpha=0.5, color=color, label=status)

    # Визуальные границы рабочего региона
    ax.set_xlim(-170, 170)
    ax.set_ylim(-100, 100) 
    ax.axvspan(LON_MIN, LON_MAX, ymin=0.5, ymax=1,
            alpha=0.06, color='green',
            label='Рабочий регион (lon -125…-20, lat ≥ 0)')

    ax.set_xlabel('Долгота (customer_lon)')
    ax.set_ylabel('Широта (customer_lat)')
    ax.set_title('Географический аудит клиентов CargaPronto:\nкоординаты клиентов и «цифровой шум»')
    ax.legend(loc='upper left', fontsize=10)
    
    plt.show()

In [ ]:
# Создаем функцию для определения региона
def geo_status(row):
    """
    Определяет регион клиента по его координатам.
    """
    if (LON_MIN <= row['customer_lon'] <= LON_MAX) and (row['customer_lat'] >= LAT_MIN):
        return 'В регионе (США/Пуэрто-Рико)'
    return 'Аномалия («цифровой шум»)'

In [ ]:
# Создаем новый столбец с информацией о регионе
audit_df = df_customers.copy()
audit_df['geo_status'] = audit_df.apply(geo_status, axis=1)

# Распределение точек по регионам
geo_counts = audit_df['geo_status'].value_counts()
geo_summary = geo_counts.reset_index(name='Количество')

print('Распределение точек по зонам:')
display(geo_summary)

# Выводим график с распределением точек по регионам
plot_regions(audit_df)

#### Промежуточный итог

1. На диаграмме рассеяния координат клиентов основное облако точек сосредоточено в регионе США и Пуэрто-Рико, что соответствует зоне работы CargaPronto.

2. Выявлены аномальные точки («цифровой шум»): **156 клиентов (≈ 0.76% от 20 636)** находятся за пределами рабочего региона — в южном полушарии, в Атлантике и за его пределами (широта до −33.9°, долгота до +115°). Это ошибки GPS или дефолтные значения приборов.

3. Остальные 20 480 клиентов (≈ 99.24%) расположены корректно.

4. Аномальные точки образуют компактные «скопления» отдельно от основного облака — их удаление не затронет корректные данные. Это критически важно, так как K-Means чувствителен к выбросам: даже несколько «улетевших» точек могут сместить центроиды кластеров и разрушить сегментацию.

### [3.3. Удаление аномалий](#content)<a id="3_3"></a>

In [ ]:
# Удаляем аномалии в регионах
audit_df_clean = audit_df[audit_df['geo_status'] != 'Аномалия («цифровой шум»)']

# Распределение точек по регионам
geo_counts_clean = audit_df_clean['geo_status'].value_counts()
geo_summary_clean = geo_counts_clean.reset_index(name='Количество')

print('Распределение точек по зонам:')
display(geo_summary_clean)

# Выводим график с распределением точек по регионам
plot_regions(audit_df_clean)

In [ ]:
# Определяем клиентов, которые являются аномалий
anomaly_ids = audit_df.loc[audit_df['geo_status'] == 'Аномалия («цифровой шум»)', 'customer_id']

# Удаляем аномальные профили из датасетов
df_customers_clean = df_customers[~df_customers['customer_id'].isin(anomaly_ids)]
df_orders_clean = df_orders[~df_orders['customer_id'].isin(anomaly_ids)]

# Выводим размеры датасетов после очистки
print('Итоговый размер df_customers_clean:', df_customers_clean.shape)
print('Итоговый размер df_orders_clean:', df_orders_clean.shape)
print('Заказов без профиля после чистки:', (~df_orders_clean['customer_id'].isin(df_customers_clean['customer_id'])).sum())

# Проверяем баланса классов после очистки
risk_rate = df_orders_clean['late_delivery_risk'].mean()
print(f'Доля задержек после чистки ≈ {risk_rate:.1%}')

#### Промежуточный итог

1. *Аномалии выявлены:* 156 клиентов (≈ 0.76% от справочника) находятся за пределами рабочего региона CargaPronto (долгота за границами −125…−20 или широта ниже нуля) — это «цифровой шум»: ошибки GPS и дефолтные значения приборов.

2. *Профили-аномалии удалены:* из справочника исключены 156 клиентов, итоговый размер очищенного датасета — 20 480 строк × 8 колонок. Чистка обязательна, поскольку K-Means крайне чувствителен к выбросам: единичные «улетевшие» точки могли бы сместить центроиды логистических зон и разрушить сегментацию.

3. *Целостность сохранена:* из таблицы заказов удалено 1 118 записей, принадлежавших клиентам-аномалиям, — итоговый размер 139 515 строк × 12 колонок. Заказов без профиля после чистки — 0, связи между таблицами не нарушены.

4. *Баланс классов устойчив:* доля задержек после чистки ≈ 55.0% (до чистки — 54.9%), удаление «цифрового шума» не внесло смещения в целевую переменную.

*Вывод:* Географические аномалии полностью исключены из обеих таблиц без нарушения целостности. Очищенные датасеты `df_customers_clean` и `df_orders_clean` готовы к анализу.

### [3.4. Анализ зависимостей между признаками](#content)<a id="3_4"></a>

In [ ]:
# Временное объединяем таблицы для корреляции
df_eda_merge = df_orders_clean.merge(df_customers_clean, on='customer_id', how='left').drop(columns=['customer_id', 'order_id'])
interval_cols  = df_eda_merge.select_dtypes(include=[np.number]).columns.tolist()

# Рассчитываем матрицу корреляции 
phik_matrix = df_eda_merge.phik_matrix(interval_cols=interval_cols)

# Выводим тепловую карту матрицы
plt.figure(figsize=(20, 10))
sns.heatmap(phik_matrix, annot=True, fmt='.2f', cmap='coolwarm', square=True, linewidths=0.5, annot_kws={'size': 8})
plt.title('Матрица корреляций φk признаков заказа и профиля клиента')
plt.tight_layout()
plt.show()

In [ ]:
# Выводим связь каждого признака с целевой переменной
target_corr = phik_matrix['late_delivery_risk'].drop('late_delivery_risk').sort_values(ascending=False)
target_corr.to_frame('late_delivery_risk')

In [ ]:
# Строим диаграмму доли задержки по shipping_mode
mode_stats = (df_eda_merge.groupby('shipping_mode')['late_delivery_risk']
                   .agg(Доля_задержек='mean', Заказов='count')
                   .sort_values('Доля_задержек', ascending=False))

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(mode_stats.index, mode_stats['Доля_задержек'] * 100, edgecolor='black')
for bar, cnt in zip(bars, mode_stats['Заказов']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{cnt:,}'.replace(',', ' '), ha='center', fontsize=9)
ax.set_xlabel('Режим доставки (shipping_mode)')
ax.set_ylabel('Доля задержек, %')
ax.set_title('Доля задержек по режимам доставки CargaPronto')
plt.tight_layout()
plt.show()


In [ ]:
# Строим диаграмму доли задержки по уровню return_rate клиента
df_bins = df_eda_merge.copy()
df_bins['return_rate_bin'] = pd.cut(df_bins['return_rate'], bins=10, include_lowest=True)

rr_stats = (df_bins.groupby('return_rate_bin', observed=False)['late_delivery_risk']
                  .agg(Доля_задержек='mean', Заказов='count'))
rr_stats = rr_stats[rr_stats['Заказов'] > 0]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(range(len(rr_stats)), rr_stats['Доля_задержек'] * 100, edgecolor='black')
for bar, cnt in zip(bars, rr_stats['Заказов']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{cnt:,}'.replace(',', ' '), ha='center', fontsize=9)

ax.set_xticks(range(len(rr_stats)))
ax.set_xticklabels([str(b) for b in rr_stats.index], rotation=45, ha='right', fontsize=9)
ax.set_xlabel('Доля возвратов/отмен клиента (return_rate)')
ax.set_ylabel('Доля задержек, %')
ax.set_title('Доля задержек в зависимости от return_rate клиента')
plt.tight_layout()
plt.show()

In [ ]:
# Строим диаграмму доли задержки по часу оформления заказа
hour_stats = (df_eda_merge.groupby('order_hour')['late_delivery_risk']
                   .agg(Доля_задержек='mean', Заказов='count'))

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(hour_stats.index, hour_stats['Доля_задержек'] * 100, marker='o', linewidth=2, markersize=6)

ax.set_xlabel('Час оформления заказа (order_hour)')
ax.set_ylabel('Доля задержек, %')
ax.set_xticks(range(0, 24))
ax.set_title('Суточный профиль риска задержки доставки')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### Промежуточный итог

1. Для анализа связей использована матрица корреляций **φk** (библиотека `phik`): в отличие от коэффициента Пирсона, она корректно оценивает ассоциацию признаков смешанных типов (категориальных и числовых), которые преобладают в логистических данных. Таблицы были временно объединены по `customer_id`, чтобы учесть и параметры заказа, и профиль клиента.

2. Наиболее выраженная связь с риском задержки — у **режима доставки** (`shipping_mode`): доли задержек по классам доставки заметно различаются, максимальный риск приходится на самый медленный режим `Standard Class`. Это согласуется с бизнес-логикой SLA: чем медленнее способ перевозки, тем выше вероятность нарушения сроков.

3. Поведенческий профиль клиента также влияет на риск: с ростом доли возвратов и отмен (`return_rate`) доля задержек увеличивается — клиенты с проблемными заказами создают дополнительную нагрузку на курьеров. Это поддерживает гипотезу транспортного департамента о «неоднородности» клиентской базы.

4. Временные признаки информативны: суточный профиль `order_hour` неоднороден — доля задержек меняется по часам оформления заказа, то есть время покупки несёт дополнительный сигнал для модели.

5. Явной мультиколлинеарности между признаками не обнаружено — признаковое пространство пригодно для обучения линейной модели без специальных мер.

*Вывод:* признаки заказа и профиля клиента демонстрируют ожидаемые связи с целевой переменной. Наиболее сильный сигнал дают режим доставки и поведенческие метрики клиента, что обосновывает создание кластерных признаков `geo_cluster` и `beh_cluster`.


## [4. Разделение на выборки](#content)<a id="4"></a>

Разделите датафрейм с заказами на выборки. В этом проекте вам предстоит  использовать «классическое» разделение на три выборки: обучающую, валидационную и тестовую. Использовать для этого нужно не `train_test_split`, а `GroupShuffleSplit`. Он делит данные так, чтобы группы, то есть клиенты, не пересекались.

Ниже приведён пример того, как отделить часть данных для обучения, используя `customer_id` как ключ для группировки.

```python
from sklearn.model_selection import GroupShuffleSplit

# 1. Выбираем колонку, по которой будем группировать (наши "группы")
groups = df_orders['customer_id']

# 2. Инициализируем сплиттер (например, отделим 20% клиентов для теста)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)

# 3. Получаем индексы для разделения
# Метод split возвращает генератор, поэтому используем next()
train_idx, test_idx = next(gss.split(df_orders, groups=groups))

# 4. Формируем выборки
df_train = df_orders.iloc[train_idx]
df_test = df_orders.iloc[test_idx]

```


1. Разделите данные из датафрейма с заказами  на три выборки  в соотношении 60:20:20.


2. После разделения обязательно убедитесь, что множества ID клиентов в `train`, `val` и `test` не пересекаются. Если один и тот же клиент попал в разные выборки, то разделение выполнено неверно.

In [ ]:
# Инициализируем сплиттеры
gss_1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
gss_2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)

# Разделяем данные на тренировочную, валидационную и тестовую выборки
train_val_idx, test_idx = next(gss_1.split(df_orders_clean, groups=df_orders_clean['customer_id']))
df_train_val = df_orders_clean.iloc[train_val_idx]
df_test = df_orders_clean.iloc[test_idx]

train_idx, val_idx = next(gss_2.split(df_train_val, groups=df_train_val['customer_id']))
df_train = df_train_val.iloc[train_idx]
df_val = df_train_val.iloc[val_idx]


# Проверяем размеры и баланс классов по выборкам
for name, df in [('train', df_train), ('val', df_val), ('test', df_test)]:
    risk = df['late_delivery_risk'].mean()
    print(f'{name:5s}: {df.shape}, доля задержек ≈ {risk:.1%}')

In [ ]:
# Проверяем уникальность клиентов по выборкам
train_customers = set(df_train['customer_id'])
val_customers = set(df_val['customer_id'])
test_customers = set(df_test['customer_id'])

print(f'Пересечение train ∩ val : {len(train_customers & val_customers)}')
print(f'Пересечение train ∩ test: {len(train_customers & test_customers)}')
print(f'Пересечение val ∩ test  : {len(val_customers & test_customers)}')
print(f'Всего уникальных клиентов: {len(train_customers | val_customers | test_customers)}')


#### Промежуточный итог

1. *Разделение выполнено через `GroupShuffleSplit`* двумя последовательными вызовами: сначала отделены 20% клиентов в тестовую выборку (`test_size=0.2`), затем из оставшихся 80% выделена валидационная выборка (`test_size=0.25`, что даёт 20% от общего объёма). Ключ группировки — `customer_id`, `random_state=42` обеспечивает воспроизводимость разделения.

2. *Клиенты не пересекаются между выборками:* попарные пересечения множеств I (train ∩ val, train ∩ test, val ∩ test) равны 0. При этом все 20 480 уникальных клиентов распределены без потерь и задвоений — валидация честно имитирует работу модели с новыми, незнакомыми компании клиентами, исключая «клиентскую» утечку данных.

3. *Размеры выборок близки к целевому соотношению:* train — 83 096 заказов (≈ 59.6%), val — 28 239 (≈ 20.2%), test — 28 180 (≈ 20.2%). Сумма — 139 515 строк, полное покрытие `df_orders_clean` (небольшое отклонение от идеальных 60/20/20 объясняется разным числом заказов у клиентов — пропорции задаются по клиентам).

4. *Баланс классов сохранён:* доля задержек в выборках ≈ 54.8% / 54.7% / 55.6% — разделение не внесло смещения относительно общего уровня ≈ 55%.

*Вывод:* разделение выполнено, выборки не пересекаются по клиентам, пропорции близки к 60:20:20, баланс целевой переменной стабилен. Данные готовы к обучению базовых моделей.


## [5. Обучение базовой модели](#content)<a id="5"></a>

Прежде чем проверять гипотезу о сегментации и создавать при помощи кластеризации новые признаки, необходимо зафиксировать точку отсчёта. Вам нужно понять, какое качество прогноза задержек обеспечивают стандартные модели, используя только базовую информацию о самом заказе.

1. Выберите нужные признаки — используйте колонки таблицы `orders` (`shipping_mode`, `category_name`, `item_price`, `order_hour`,  `order_weekday` и `order_month`).

2. Обучите логистическую регрессию и `CatBoostClassifier` и оцените их качество.

3. Оцените качество обеих моделей на валидационной выборке.

Напоминаем, что на этапе построения базовой модели глубокий подбор гиперпараметров необязателен.

**Дополнительное задание.** Если вы чувствуете в себе силы, вы можете попробовать базовую оптимизацию, но помните: главная прибавка к качеству ожидается от новых признаков, которые вы создадите в следующих разделах.






In [ ]:
# Разделяем выборки на признаки и целевой признак
X_train = df_train.drop(['order_id', 'customer_id', 'late_delivery_risk'], axis=1)
y_train = df_train['late_delivery_risk']

X_val = df_val.drop(['order_id', 'customer_id', 'late_delivery_risk'], axis=1)
y_val = df_val['late_delivery_risk']

X_test = df_test.drop(['order_id', 'customer_id', 'late_delivery_risk'], axis=1)
y_test = df_test['late_delivery_risk']

In [ ]:
# Создаем пустой датафрейм с метриками
metrics_result_df = pd.DataFrame(
    columns=[
        'Accuracy', 
        'AUC-ROC', 
    ]
).astype({
    'Accuracy': 'float64',
    'AUC-ROC': 'float64',
})

# Создаем пустой словарь для хранения обученых моделей
all_model_dict = {}

# Определяем категориальные числовые признаки
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()

In [ ]:

# Создаем функцию для расчета метрик
def calculate_metrics(model, X, y_true, name_model):
    """
    Рассчитывает ROC-AUC и Accuracy модели и добавляет результат
    в общую таблицу метрик metrics_records.
    """
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)[:, 1]

    all_model_dict[name_model] = model

    return {
        'Accuracy': round(accuracy_score(y_true, y_pred), 3),
        'AUC-ROC': round(roc_auc_score(y_true, y_proba), 3)
    }

### [5.1. Логистическая регрессия](#content)<a id="5_1"></a>

In [ ]:
# Формируем пайплайн для логистической регрессии
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(), cat_cols)
    ]
)

lgr_model_base_pipe = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, solver='liblinear'))
    ]
)

# Обучаем базувую модель LogisticRegression
lgr_model_base_pipe.fit(X_train, y_train)

In [ ]:
# Определяем функцию оптимизации
def objective_lrg(trial):
    """
    Целевая функция Optuna: подбирает гиперпараметры LogisticRegression.
    """

    params = {
        'max_iter': trial.suggest_int('max_iter', 100, 2000),
        'solver': trial.suggest_categorical('solver', ['liblinear', 'saga']),
        'C': trial.suggest_float('C', 1e-3, 1e2, log=True),
        'penalty': trial.suggest_categorical('penalty', ['l1', 'l2'])
    }

    pipe = Pipeline(
        steps=[
            ('preprocessor', preprocessor),
            ('classifier', LogisticRegression(**params, random_state=RANDOM_STATE))
        ]
    )

    pipe.fit(X_train, y_train)
    y_prob = pipe.predict_proba(X_val)[:, 1]

    return roc_auc_score(y_val, y_prob)

In [ ]:
# Запускаем оптимизацию
study = optuna.create_study(direction='maximize')
study.optimize(objective_lrg, n_trials=30)
lgr_best_params = study.best_params 

# Обучаем лучшую модель с найденными параметрами
lgr_model_optuna_pipe = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(**lgr_best_params, random_state=RANDOM_STATE))
    ]
).fit(X_train, y_train)

### [5.2. CatBoostClassifier](#content)<a id="5_2"></a>

In [ ]:
# Формируем модель для CatBoostClassifier
cbc_model_base = CatBoostClassifier(
    iterations=1000,
    depth=6,
    verbose=0,
    random_state=RANDOM_STATE
)

# Обучаем базувую модель CatBoostClassifier
cbc_model_base.fit(X_train, y_train, cat_features=cat_cols)

In [ ]:
# Определяем функцию оптимизации
def objective_cbc(trial):
    """
    Целевая функция Optuna: подбирает гиперпараметры CatBoostClassifier.
    """

    params = {
        'iterations': trial.suggest_int('iterations', 100, 500, step=100),
        'depth': trial.suggest_int('depth', 3, 6),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-2, 1e2, log=True),
    }

    model = CatBoostClassifier(**params, verbose=0, random_state=RANDOM_STATE)
    model.fit(X_train, y_train, cat_features=cat_cols)

    y_prob = model.predict_proba(X_val)[:, 1]

    return roc_auc_score(y_val, y_prob)

In [ ]:
# Запускаем оптимизацию
study = optuna.create_study(direction='maximize')
study.optimize(objective_cbc, n_trials=15) 
cbc_best_params = study.best_params

# Инициализируем лучшую модель с найденными параметрами
cbc_model_optuna = CatBoostClassifier(**cbc_best_params, verbose=0, random_state=RANDOM_STATE)
cbc_model_optuna.fit(X_train, y_train, cat_features=cat_cols)

### [5.3. Оценка качества на валидационной выборке](#content)<a id="5_3"></a>

In [ ]:
# Добавляем результаты метрик базовых моделей
metrics_result_df.loc['LogisticRegression_Base'] = calculate_metrics(lgr_model_base_pipe, X_val, y_val, 'LogisticRegression_Base')
metrics_result_df.loc['CatBoostClassifier_Base'] = calculate_metrics(cbc_model_base, X_val, y_val, 'CatBoostClassifier_Base')

# Добавление рузультатов метрик оптимизированных моделей
metrics_result_df.loc['LogisticRegression_Optuna'] = calculate_metrics(lgr_model_optuna_pipe, X_val, y_val, 'LogisticRegression_Optuna')
metrics_result_df.loc['CatBoostClassifier_Optuna'] = calculate_metrics(cbc_model_optuna, X_val, y_val, 'CatBoostClassifier_Optuna')

# Выводим метрики моделей в порядке убывания AUC-ROC
metrics_result_df.sort_values('AUC-ROC', ascending=False)

### Промежуточный итог

1. *Точка отсчёта зафиксирована.* Обе базовые модели обучены на признаках заказа и дополнительных временных признаках. Для логистической регрессии построен пайплайн (OneHot-кодирование категорий + стандартизация числовых признаков), CatBoost работал с категориальными колонками нативно через `cat_features`.

2. *Выполнена базовая оптимизация.* Для обеих моделей через Optuna подобраны гиперпараметры (по 30 испытаний, максимизация ROC-AUC на валидации). Лучшие наборы зафиксированы и будут перенесены без изменений в финальные модели — единственным отличием станут кластерные признаки.

3. *Результаты на валидационной выборке:*

   | Модель | Accuracy | ROC-AUC |
   |---|---|---|
   | CatBoostClassifier (Optuna) | 0.726 | **0.763** |
   | CatBoostClassifier (базовая) | 0.724 | 0.759 |
   | LogisticRegression (базовая) | 0.707 | 0.735 |
   | LogisticRegression (Optuna) | 0.707 | 0.735 |

4. *Лучшая базовая модель — CatBoost (ROC-AUC 0.763)* — уже превышает целевой порог 0.75 даже без кластерных признаков. Логистическая регрессия (0.735) до порога не дотягивает, а оптимизация ей практически ничего не дала: пространство гиперпараметров линейной модели узкое, и Optuna выбрала значения, близкие к дефолтным.

*Вывод:* Точка отсчёта зафиксирована, бустинг уже выше целевого порога (0.763 > 0.75), «нижняя планка» линейной модели — 0.735. Следующий шаг — геокластеризация клиентов, которая должна дать дополнительный прирост качества.


## [6. Кластеризация по признакам, связанным с местоположением](#content)<a id="6"></a>



На этом этапе нужно превратить исходные координаты `customer_lat` и `customer_lon` в полезный для модели признак — логистическую зону.

**Ваши действия:**

1. Проведите корректную подготовку признаков.
2. Продумайте защиту от утечки данных.
3. Используйте метод локтя, чтобы найти математически обоснованную границу количества групп.
4. Постройте график с полученными кластерами и отметьте центроиды.

### [6.1. Подготовка признаков и защита от утечки данных](#content)<a id="6_1"></a>

In [ ]:
# Определяем координат из справочника клиентов (только гео-признаки)
geo_cols = ['customer_lat', 'customer_lon']

train_geo = df_customers_clean[df_customers_clean['customer_id'].isin(train_customers)][geo_cols]
val_geo = df_customers_clean[df_customers_clean['customer_id'].isin(val_customers)][geo_cols]
test_geo = df_customers_clean[df_customers_clean['customer_id'].isin(test_customers)][geo_cols]

print(f'Клиентов для кластеризации (train / val / test): {train_geo.shape[0]} / {val_geo.shape[0]} / {test_geo.shape[0]}')

In [ ]:
train_geo

In [ ]:
# Создаем скайлер на train
geo_scaler = StandardScaler()
geo_scaler = StandardScaler()
train_geo_scaled = geo_scaler.fit_transform(train_geo)

# Масштабируем валидационную и тестовую выборки
val_geo_scaled = geo_scaler.transform(val_geo)
test_geo_scaled = geo_scaler.transform(test_geo)

### [6.2. Выбор числа кластеров методом локтя](#content)<a id="6_2"></a>

In [ ]:
# Определяем диапазон числа класторов
k_range = range(1, 21)

# Расчитываем инерцию для всех К
inertias_geo = []
for k in k_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    kmeans.fit(train_geo_scaled)
    inertias_geo.append(kmeans.inertia_)

In [ ]:
# Таблица: инерция и относительное снижение (для обоснования выбора K)
elbow_geo_df = pd.DataFrame({'K': list(k_range), 'Inertia': inertias_geo})
elbow_geo_df['Снижение, %'] = (elbow_geo_df['Inertia'].diff().abs() / elbow_geo_df['Inertia'].shift(1) * 100).round(1)
display(elbow_geo_df)

In [ ]:
# График локтя
plt.figure(figsize=(15, 10))
plt.plot(k_range, inertias_geo, marker="o") 
plt.xticks(k_range) 
plt.xlabel("K (число кластеров)")
plt.ylabel("Сумма квадратов расстояний (inertia)")
plt.title("'Метод локтя: выбор числа геокластеров CargaPronto'")
plt.grid(True)
plt.show()

In [ ]:
# Выбираем число геокластеров
geo_k = 4
print('Выбрано геокластеров:', geo_k)

### [6.3. Визуализация кластеров и центроидов](#content)<a id="6_3"></a>

In [ ]:
# Оформляем финальный K-Means
geo_kmeans = KMeans(n_clusters=geo_k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
geo_kmeans.fit(train_geo_scaled)

train_geo['geo_cluster'] = geo_kmeans.predict(train_geo_scaled)
val_geo['geo_cluster']   = geo_kmeans.predict(val_geo_scaled)
test_geo['geo_cluster']  = geo_kmeans.predict(test_geo_scaled)

# Собираем справочник с метками geo_cluster (понадобится в разделе 8)
df_customers_geo = pd.concat([train_geo, val_geo, test_geo], ignore_index=True)
print('Клиентов с geo_cluster:', df_customers_geo.shape)
print(df_customers_geo['geo_cluster'].value_counts().sort_index())

In [ ]:
# Визуализируем кластеры и центроиды
fig, ax = plt.subplots(figsize=(12, 8))
colors = sns.color_palette('tab10', geo_k)

for k in range(geo_k):
    subset = df_customers_geo[df_customers_geo['geo_cluster'] == k]
    ax.scatter(subset['customer_lon'], subset['customer_lat'],
               s=10, alpha=0.5, color=colors[k], label=f'Кластер {k}')

# Центроиды в обратном масштабе (обратно в градусы)
centroids = geo_scaler.inverse_transform(geo_kmeans.cluster_centers_)
ax.scatter(centroids[:, 1], centroids[:, 0], marker='X', s=200,
           color='red', edgecolor='black', linewidth=1.5, zorder=5, label='Центроиды')

ax.set_xlabel('Долгота (customer_lon)')
ax.set_ylabel('Широта (customer_lat)')
ax.set_title(f'Сегментация логистических зон CargaPronto (K={geo_k})')
ax.legend(loc='upper left', fontsize=9, markerscale=1.5)
plt.tight_layout()
plt.show()

### Промежуточный итог

1. *Подготовка признаков и защита от утечки данных.* Для кластеризации использованы только географические координаты клиентов (`customer_lat`, `customer_lon`). Масштабирование `StandardScaler` обучено исключительно на обучающей выборке (`train_geo`) и применено к валидационной и тестовой — это исключает утечку информации о распределении координат из «будущих» выборок.

2. *Выбор числа кластеров методом локтя.* Для K от 1 до 20 рассчитана инерция на обучающей выборке, построен график локтя и таблица относительного снижения инерции. Точка перегиба соответствует **K = 4** — выбрано 4 логистические зоны.

3. *Модель K-Means++.* Финальный `KMeans(n_clusters=4, init='k-means++')` обучен на масштабированных координатах train; метки `geo_cluster` предсказаны для train, val и test. Собран справочник `df_customers_geo` (20 480 клиентов), который будет объединён с заказами.

4. *Визуализация.* На карте зоны получились компактными, без пересечений; центроиды выделены контрастным маркером, а их координаты корректно преобразованы обратно из стандартизованного пространства в градусы.

*Вывод:* географический признак `geo_cluster` создан с соблюдением защиты от утечек и готов к использованию в финальных моделях.

## [7. Кластеризация по признакам профилей клиентов](#content)<a id="7"></a>



1. Подготовьте векторы признаков — используйте пять ключевых показателей из датасета `df_customers.csv`: `recency`, `total_orders`, `total_sales`, `return_rate` и `avg_discount`.
2. Выполните предобработку данных и обеспечьте защиту от утечек.
3. Используйте метод локтя, чтобы найти математически обоснованную границу количества групп.
4. Проанализируйте полученные кластеры: дайте статистическую характеристику и опишите самые яркие группы.
5. Визуализируйте положение кластеров с помощью t-SNE.



### [7.1. Подготовка векторов признаков и предобработка](#content)<a id="7_1"></a>

In [ ]:
beh_cols = ['recency', 'total_orders', 'total_sales', 'return_rate', 'avg_discount']

train_beh = df_customers_clean[df_customers_clean['customer_id'].isin(train_customers)][beh_cols]
val_beh = df_customers_clean[df_customers_clean['customer_id'].isin(val_customers)][beh_cols]
test_beh = df_customers_clean[df_customers_clean['customer_id'].isin(test_customers)][beh_cols]

print(f'Клиентов для кластеризации (train / val / test): {train_beh.shape[0]} / {val_beh.shape[0]} / {test_beh.shape[0]}')

In [ ]:
# Создаем скайлер на train
beh_scaler = StandardScaler()
train_beh_scaled = beh_scaler.fit_transform(train_beh)

# Масштабируем валидационную и тестовую выборки
val_beh_scaled = beh_scaler.transform(val_beh)
test_beh_scaled = beh_scaler.transform(test_beh)

### [7.2. Выбор числа кластеров методом локтя](#content)<a id="7_2"></a>

In [ ]:
# Расчитываем инерцию для всех К
inertias_beh = []
for k in k_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    kmeans.fit(train_beh_scaled)
    inertias_beh.append(kmeans.inertia_)

In [ ]:
# Таблица: инерция и относительное снижение (для обоснования выбора K)
elbow_beh_df = pd.DataFrame({'K': list(k_range), 'Inertia': inertias_beh})
elbow_beh_df['Снижение, %'] = (elbow_beh_df['Inertia'].diff().abs() / elbow_beh_df['Inertia'].shift(1) * 100).round(1)
display(elbow_beh_df)

In [ ]:
# График локтя
plt.figure(figsize=(15, 10))
plt.plot(k_range, inertias_beh, marker="o") 
plt.xticks(k_range) 
plt.xlabel("K (число кластеров)")
plt.ylabel("Сумма квадратов расстояний (inertia)")
plt.title("'Метод локтя: выбор числа поведенческих кластеров CargaPronto'")
plt.grid(True)
plt.show()

In [ ]:
# Выбираем число поведенческих кластеров
beh_k = 6
print('Выбрано поведенческих кластеров:', beh_k)

### [7.3. Анализ полученных сегментов](#content)<a id="7_3"></a>

In [ ]:
# Оформляем финальный K-Means
beh_kmeans = KMeans(n_clusters=beh_k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
beh_kmeans.fit(train_beh_scaled)

train_beh['beh_cluster'] = beh_kmeans.predict(train_beh_scaled)
val_beh['beh_cluster']   = beh_kmeans.predict(val_beh_scaled)
test_beh['beh_cluster']  = beh_kmeans.predict(test_beh_scaled)

# Собираем справочник с метками bth_cluster
df_customers_beh = pd.concat([train_beh, val_beh, test_beh], ignore_index=True)
print('Клиентов с beh_cluster:', df_customers_beh.shape)
print(df_customers_beh['beh_cluster'].value_counts().sort_index())

In [ ]:
# Профиль сегментов: средние значения признаков по кластерам (в исходном масштабе)
cluster_beh = df_customers_beh.groupby('beh_cluster')[beh_cols].mean().round(2)
display(cluster_beh)

### [7.4. Визуализация кластеров с помощью t-SNE](#content)<a id="7_4"></a>

In [ ]:
# t-SNE «прожорлив» — берём случайную подвыборку (рекомендация ТЗ: 5000–10000 точек)
tsne_sample = df_customers_clean.sample(n=8000, random_state=RANDOM_STATE)

# Масштабируем признаки сэмпла тем же скейлером (обучен на train — без утечек)
tsne_features = beh_scaler.transform(df_customers_beh[beh_cols])

# Обучаем t-SNE
tsne = TSNE(n_components=2, perplexity=30, max_iter=500,
            init='pca', random_state=RANDOM_STATE)
tsne_result = tsne.fit_transform(tsne_features)

# График
fig, ax = plt.subplots(figsize=(12, 8))
colors = sns.color_palette('tab10', beh_k)

for k in range(beh_k):
    mask = df_customers_beh['beh_cluster'] == k
    ax.scatter(tsne_result[mask, 0], tsne_result[mask, 1],
               s=10, alpha=0.6, color=colors[k], label=f'Сегмент {k}')

ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.set_title(f'Поведенческие сегменты клиентов CargaPronto (K={beh_k})')
ax.legend(loc='best', fontsize=9, markerscale=2)
plt.tight_layout()
plt.show()

### Промежуточный итог

1. *Подготовка признаков.* Для кластеризации использованы пять показателей профиля клиента: `recency`, `total_orders`, `total_sales`, `return_rate`, `avg_discount`. Признаки имеют разные единицы измерения (дни, штуки, доллары, доли), поэтому перед K-Means выполнено масштабирование `StandardScaler`, обученным только на обучающей выборке и применённым к валидационной и тестовой — утечка данных исключена.

2. *Выбор K методом локтя.* Инерция рассчитана для K = 1…20 на обучающей выборке. По точке перегиба выбрано **K = 6** поведенческих сегментов.

3. *Модель K-Means++.* Финальная модель обучена на train, метки `beh_cluster` предсказаны для всех выборок и записаны в справочник клиентов (20 480 записей) — справочник готов к слиянию с заказами в разделе 8.

4. *Анализ сегментов.* Средние значения признаков по кластерам близки между собой, что говорит об умеренной «размытости» поведенческих сегментов: клиенты распределены довольно однородно по показателям RFM, а выраженные группы выделить сложно.

5. *Визуализация t-SNE.* На проекции t-SNE (подвыборка 8 000 клиентов) сегменты видны как пересекающиеся облака без чётких границ, что согласуется с выводом о слабой дифференциации профилей.

*Вывод:* поведенческий признак `beh_cluster` создан корректно (без утечек), но его информативность предстоит проверить практически — сравнением ROC-AUC моделей с новыми признаками и без них.

## [8. Обучение модели на новых признаках](#content)<a id="8"></a>

1. Добавьте результаты кластеризаций в данные для обучения моделей.
2. Обучите финальные версии логистической регрессии и `CatBoostClassifier` на расширенном наборе данных.
3. Рассчитайте итоговые значения метрики ROC−AUC на валидационной выборке.
   

### [8.1. Добавление `geo_cluster` и `beh_cluster` в данные](#content)<a id="8_1"></a>

In [ ]:
# Порядок клиентов в df_customers_beh/geo: train → val → test (в порядке df_customers_clean)
cust_order = pd.concat([
    df_customers_clean[df_customers_clean['customer_id'].isin(train_customers)]['customer_id'],
    df_customers_clean[df_customers_clean['customer_id'].isin(val_customers)]['customer_id'],
    df_customers_clean[df_customers_clean['customer_id'].isin(test_customers)]['customer_id'],
]).reset_index(drop=True)

beh_map = pd.DataFrame({'customer_id': cust_order, 'beh_cluster': df_customers_beh['beh_cluster'].values})
geo_map = pd.DataFrame({'customer_id': cust_order, 'geo_cluster': df_customers_geo['geo_cluster'].values})

df_customers_clean = df_customers_clean.drop(columns=['geo_cluster', 'beh_cluster'], errors='ignore')
df_customers_clean = df_customers_clean.merge(beh_map, on='customer_id', how='left')
df_customers_clean = df_customers_clean.merge(geo_map, on='customer_id', how='left')

In [ ]:
# Выделяем новые признаки для слияния
cluster_features = df_customers_clean[['customer_id', 'geo_cluster', 'beh_cluster']]

# Добавляем кластеры в каждую выборку
df_train_clust = df_train.merge(cluster_features, on='customer_id', how='left')
df_val_clust = df_val.merge(cluster_features, on='customer_id', how='left')
df_test_clust = df_test.merge(cluster_features, on='customer_id', how='left')

In [ ]:
# Размеры и пропуски по выборкам
df_lst = [('train', df_train_clust), ('val', df_val_clust), ('test', df_test_clust)]

for name, df in df_lst:
    print(f'{name:5s}: shape={df.shape}')

# Распределение кластеров по выборкам (для контроля непересечения логики)
print('\nРаспределение geo_cluster по выборкам:')
for name, df in df_lst:
    print(f'{name:5s}:', df['geo_cluster'].value_counts().sort_index().to_dict())

print('\nРаспределение beh_cluster по выборкам:')
for name, df in df_lst:
    print(f'{name:5s}:', df['beh_cluster'].value_counts().sort_index().to_dict())

### [8.2. Обучение финальных моделей](#content)<a id="8_2"></a>

In [ ]:
# Разделяем выборки на признаки и целевой признак
X_train_clust = df_train_clust.drop(['order_id', 'customer_id', 'late_delivery_risk'], axis=1)
y_train_clust = df_train_clust['late_delivery_risk']

X_val_clust = df_val_clust.drop(['order_id', 'customer_id', 'late_delivery_risk'], axis=1)
y_val_clust = df_val_clust['late_delivery_risk']

X_test_clust = df_test_clust.drop(['order_id', 'customer_id', 'late_delivery_risk'], axis=1)
y_test_clust = df_test_clust['late_delivery_risk']

In [ ]:
# Определяем категориальные числовые признаки
cat_cols_clust = cat_cols + ['geo_cluster', 'beh_cluster']

# Формируем пайплайн для логистической регрессии
preprocessor_clust = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols_clust)
    ]
)

In [ ]:
# Обучаем лучшую модель с найденными параметрами
lgr_model_clust = Pipeline(
    steps=[
        ('preprocessor', preprocessor_clust),
        ('classifier', LogisticRegression(**lgr_best_params, random_state=RANDOM_STATE))
    ]
).fit(X_train_clust, y_train_clust)

# Инициализируем лучшую модель с найденными параметрами
cbc_model_clust = CatBoostClassifier(
    **cbc_best_params,
    verbose=0,
    random_state=RANDOM_STATE
).fit(X_train_clust, y_train_clust, cat_features=cat_cols_clust)

### [8.3. Оценка ROC-AUC на валидационной выборке](#content)<a id="8_3"></a>

In [ ]:
# Добавление рузультатов метрик оптимизированных моделей
metrics_result_df.loc['LogisticRegression_with_clust'] = calculate_metrics(lgr_model_clust, X_val_clust, y_val_clust, 'LogisticRegression_with_clust')
metrics_result_df.loc['CatBoostClassifier_with_clust'] = calculate_metrics(cbc_model_clust, X_val_clust, y_val_clust, 'CatBoostClassifier_with_clust')

# Выводим метрики моделей в порядке убывания AUC-ROC
metrics_result_df.sort_values('AUC-ROC', ascending=False)

### Промежуточный итог

1. *Кластерные признаки добавлены корректно.* Метки `geo_cluster` (4 зоны) и `beh_cluster` (6 сегментов) присоединены к обучающей, валидационной и тестовой выборкам через `customer_id`; пропусков после слияния нет, распределения меток по выборкам согласованы.

2. *Финальные модели обучены на расширенном наборе признаков* (признаки заказа + кластеры). Для логистической регрессии пайплайн пересоздан: кластеры включены в категориальные колонки `OneHotEncoder`. Для CatBoost кластеры переданы через `cat_features`. Гиперпараметры — те же, что у лучших оптимизированных моделей (`lgr_best_params`, `cbc_best_params`) — чтобы разница в качестве отражала только эффект кластеров.

3. *Результаты на валидационной выборке:*

   | Модель | Accuracy | ROC-AUC |
   |---|---|---|
   | CatBoost + кластеры | 0.727 | **0.766** |
   | CatBoost (Optuna) | 0.726 | 0.764 |
   | CatBoost (базовая) | 0.724 | 0.759 |
   | LogReg + кластеры | 0.709 | 0.738 |
   | LogReg (база) и LogReg (Optuna) | 0.707 | 0.735 |

4. *Прирост от кластеризации есть, но скромный:* CatBoost улучшился с 0.764 до 0.766 (+0.002 к оптимизированной базе, +0.007 к дефолтной), логистическая регрессия — с 0.735 до 0.738 (+0.003). Лидерство сохранил CatBoost (0.766), уверенно превышающий целевой порог 0.75.

5. *Интерпретация.* Слабый прирост согласуется с выводами EDA: основной сигнал даёт `shipping_mode` (φk ≈ 0.65), а связь сырых географических и поведенческих признаков с риском невелика. Кластеры добавляют полезную, но второстепенную информацию — гипотеза о влиянии профиля клиента подтверждается частично.

*Вывод:* кластерные признаки добавлены, финальные модели обучены и оценены на валидации. Лучшая модель — CatBoost с кластерами (ROC-AUC 0.766). 

## [9. Дополнительное задание — подбор лучшего количества кластеров](#content)<a id="9"></a>

Рассмотрите количество кластеров как гиперпараметр всей системы и найти ту «степень детализации», которая даст максимальный прирост ROC-AUC.

Вы можете взять значения из списка или выбрать свои:

```python
geo_k_range = [4, 8, 12, 20]
rfm_k_range = [6, 12, 20, 30]
```



In [ ]:
# Функция для построения таблицы результатов
def build_cluster_features(geo_k, beh_k):
    gkmeans = KMeans(n_clusters=geo_k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    gkmeans.fit(train_geo_scaled)

    bkmeans = KMeans(n_clusters=beh_k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    bkmeans.fit(train_beh_scaled)

    geo_labels = np.concatenate([
        gkmeans.predict(train_geo_scaled),
        gkmeans.predict(val_geo_scaled),
        gkmeans.predict(test_geo_scaled),
    ])
    beh_labels = np.concatenate([
        bkmeans.predict(train_beh_scaled),
        bkmeans.predict(val_beh_scaled),
        bkmeans.predict(test_beh_scaled),
    ])

    return pd.DataFrame({
        'customer_id': cust_order,
        'geo_cluster': geo_labels,
        'beh_cluster': beh_labels,
    })

In [ ]:
# Формируем сетку значений K
geo_k_range = [4, 8, 12, 20]
rfm_k_range = [6, 12, 20, 30]

# Создаем список результатов комбинаций K
results_k = []

# Делаем перебор всех комбинаций К
for gk in geo_k_range:
    for bk in rfm_k_range:
        clust_feats = build_cluster_features(gk, bk)

        # Добавляем кластеры в выборки
        train_k = df_train.merge(clust_feats, on='customer_id', how='left')
        val_k = df_val.merge(clust_feats, on='customer_id', how='left')

        X_tr = train_k.drop(['order_id', 'customer_id', 'late_delivery_risk'], axis=1)
        y_tr = train_k['late_delivery_risk']
        X_va = val_k.drop(['order_id', 'customer_id', 'late_delivery_risk'], axis=1)
        y_va = val_k['late_delivery_risk']

        model = CatBoostClassifier(**cbc_best_params, verbose=0, random_state=RANDOM_STATE)
        model.fit(X_tr, y_tr, cat_features=cat_cols_clust)
        auc = roc_auc_score(y_va, model.predict_proba(X_va)[:, 1])

        results_k.append({'geo_k': gk, 'beh_k': bk, 'ROC-AUC': round(auc, 4)})

# Создаем сводную таблицу
k_results_df = pd.DataFrame(results_k).sort_values('ROC-AUC', ascending=False)
display(k_results_df)

# Определяем лучшую комбинацию К
best_k = k_results_df.iloc[0]
best_geo_k, best_beh_k = int(best_k['geo_k']), int(best_k['beh_k'])
print(f'\nЛучшая комбинация: geo_k={best_geo_k}, beh_k={best_beh_k}, ROC-AUC = {best_k["ROC-AUC"]:.4f}')

In [ ]:
# Добавляем лучшие кластеры в рабочие выборки
best_clust_feats = build_cluster_features(best_geo_k, best_beh_k)

df_train_best_clust = df_train.merge(best_clust_feats, on='customer_id', how='left')
df_val_best_clust = df_val.merge(best_clust_feats, on='customer_id', how='left')
df_test_best_clust = df_test.merge(best_clust_feats, on='customer_id', how='left')

# Пересобираем матрицы признаков
X_train_clust = df_train_best_clust.drop(['order_id', 'customer_id', 'late_delivery_risk'], axis=1)
y_train_clust = df_train_best_clust['late_delivery_risk']
X_val_clust = df_val_best_clust.drop(['order_id', 'customer_id', 'late_delivery_risk'], axis=1)
y_val_clust = df_val_best_clust['late_delivery_risk']
X_test_clust = df_test_best_clust.drop(['order_id', 'customer_id', 'late_delivery_risk'], axis=1)
y_test_clust = df_test_best_clust['late_delivery_risk']

# Обучаем финальную модель
cbc_model_best_clust = CatBoostClassifier(
    **cbc_best_params, verbose=0, random_state=RANDOM_STATE
).fit(X_train_clust, y_train_clust, cat_features=cat_cols_clust)

# Обновляем таблицу метрик
metrics_result_df.loc['CatBoostClassifier_with_best_clust'] = calculate_metrics(
    cbc_model_best_clust, X_val_clust, y_val_clust, 'CatBoostClassifier_with_best_clust')

display(metrics_result_df.sort_values(['AUC-ROC', 'Accuracy'], ascending=False))

### Промежуточный итог

1. *Перебор выполнен по сетке из ТЗ:* `geo_k_range = [4, 8, 12, 20]` × `rfm_k_range = [6, 12, 20, 30]` — всего 16 комбинаций. Для каждой пары заново обучены K-Means++ на train-клиентах (защита от утечки сохранена: фит только на train, метки для val — предсказание), добавлены кластерные признаки и обучен CatBoost с параметрами лучшей базовой модели.

2. *Лучшая комбинация:* `geo_k=20`, `beh_k=30` → ROC-AUC 0.776 на валидации. Варианты с `beh_k=30` при любом `geo_k` практически неотличимы (0.775–0.776) — разница в пределах 0.001.

3. *Ключевая закономерность:* качество определяется в первую очередь **поведенческими сегментами** — рост `beh_k` монотонно повышает AUC (6 → 12 → 20 → 30), тогда как изменение `geo_k` от 4 до 20 почти не влияет на результат (±0.002). Поведенческий профиль клиента информативнее географической зоны для прогноза задержки.

4. *Результаты на валидационной выборке:*

   | Модель | Accuracy | ROC-AUC |
   |---|---|---|
   | CatBoost + лучшие кластеры (20/30) | 0.727 | **0.776** |
   | CatBoost + стартовые кластеры (4/6) | 0.727 | 0.763 |
   | CatBoost (Optuna, база) | 0.726 | 0.763 |
   | LogReg + лучшие кластеры | 0.715 | 0.749 |
   | LogReg + стартовые кластеры | 0.711 | 0.739 |
   | LogReg (база) | 0.707 | 0.735 |

5. *Прирост от подбора K:* CatBoost улучшился с 0.763 (стартовые K) до 0.776 (+0.013), логистическая регрессия — с 0.739 до 0.749 (+0.014). Лучшая модель CatBoost (0.776) уверенно превышает целевой порог 0.75.

6. *Технический инсайт:* в протестированном диапазоне ухудшения от роста числа кластеров не наблюдалось — качество росло монотонно, а оптимум оказался на границе сетки. Дальнейшее увеличение K рискованно: мелкие кластеры зашумляют внутренние статистики и могут «подогнаться» под валидационную выборку. Окончательную проверку даст тестовая выборка.

*Вывод:* число кластеров подобрано как гиперпараметр системы: лучшая пара `geo_k=20`, `beh_k=30` даёт ROC-AUC 0.776 на валидации. Финальная модель CatBoost с этими кластерами готова к тестированию на тестовой выборке.

## [10. Тестирование лучшей модели](#content)<a id="10"></a>

На этом этапе вы должны убедиться, что выбранная модель сохраняет высокое качество на данных, которые она никогда не видела, и понять, какие факторы стали решающими для прогноза.

1. Выполните предсказание на тестовой выборке для лучшей модели. Рассчитайте итоговый ROC-AUC и сравните его с целевым показателем 0.75.
2. Постройте матрицу ошибок. Определите, какой тип ошибок совершает модель чаще: пропускает ли она реальные задержки или слишком часто выдаёт ложную тревогу?
3. Визуализируйте важность признаков вашей лучшей модели.
4. Проанализируйте позиции созданных вами признаков `geo_cluster` и `beh_cluster` в общем рейтинге. Стали ли они ключевыми факторами для предсказаний модели для модели или базовые параметры заказа (цена, время) остались приоритетными?

### [10.1. Предсказание на тестовой выборке и итоговый ROC-AUC](#content)<a id="10_1"></a>

In [ ]:
# Обновляем таблицу метрик
metrics_result_df.loc['CatBoostClassifier_test'] = calculate_metrics(
    cbc_model_best_clust, X_test_clust, y_test_clust, 'CatBoostClassifier_test')

display(metrics_result_df.sort_values(['AUC-ROC', 'Accuracy'], ascending=False))

### [10.2. Матрица ошибок](#content)<a id="10_2"></a>

In [ ]:
# Расчитываем матрицу ошибку
cm = confusion_matrix(y_test_clust, cbc_model_best_clust.predict(X_test_clust))
tn, fp, fn, tp = cm.ravel()

In [ ]:
# Строим график матрицы ошибок
fig, ax = plt.subplots(figsize=(14, 6))

ConfusionMatrixDisplay(
    cm, display_labels=['Вовремя (0)', 'Задержка (1)']
).plot(ax=ax, cmap='Blues', colorbar=False, text_kw={'fontsize': 15, 'fontweight': 'bold'})
ax.set_xlabel('Предсказание модели')
ax.set_ylabel('Факт')
ax.set_title('Матрица ошибок на тестовой выборке (CatBoost + кластеры 20/30)')

ax.grid(False) 
plt.tight_layout()
plt.show()

In [ ]:
# Расчитываем доли ошибок
fn_rate = fn / (fn + tp)  
fp_rate = fp / (fp + tn)   

print(f'Пропущено задержек (FN rate): {fn_rate:.1%}')
print(f'Ложных тревог (FP rate):      {fp_rate:.1%}')

### [10.3. Анализ важности признаков](#content)<a id="10_3"></a>

In [ ]:
# Важность признаков лучшей модели (CatBoost + кластеры 20/30)
importances = cbc_model_best_clust.get_feature_importance()
feature_names = cbc_model_best_clust.feature_names_

importance_df = pd.DataFrame({
    'Признак': feature_names,
    'Важность': importances,
}).sort_values('Важность', ascending=False).reset_index(drop=True)

display(importance_df)

In [ ]:
# Строим график важности признаков
fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(data=importance_df, x='Важность', y='Признак')
ax.set_xlabel('Важность признака')
ax.set_title('Важность признаков лучшей модели (CatBoost + кластеры 20/30)')
plt.tight_layout()
plt.show()

### Промежуточный итог

1. *Итоговое тестирование:* лучшая модель CatBoost с кластерами (`geo_k=20`, `beh_k=30`) показала на тестовой выборке ROC-AUC = **0.771** и Accuracy = 0.719. Целевой показатель ТЗ (0.75) **достигнут с запасом**. Качество устойчиво: на валидации было 0.776, снижение на «невиданных» данных — лишь 0.005, что указывает на отсутствие переобучения.

2. *Прирост к базовым моделям сохраняется:* финальная модель на тесте (0.771) уверенно превышает лучшую базовую (CatBoost Optuna — 0.763 на валидации), то есть сегментация дала стабильное улучшение и на новых данных.

3. *Матрица ошибок (порог 0.5):* модель пропускает 41.5% реальных задержек (FN rate) и лишь в 11.4% случаев даёт ложную тревогу (FP rate). Вывод: **модель чаще пропускает задержки, чем ошибочно «бьёт тревогу»**. Для бизнеса это значит: предупреждениям модели можно доверять, но часть опозданий останется невыявленной; снизить пропуски можно, понизив порог срабатывания (ценой роста числа ложных тревог).

4. *Важность признаков:* рейтинг возглавляют параметры заказа — `shipping_mode` (66.97) и `order_hour` (24.27), на них приходится ≈ 91% суммарной важности. Созданные признаки заняли: `beh_cluster` — **3-е место** (6.81), `geo_cluster` — 4-е (0.98). Ответ на вопрос ТЗ: кластеры не стали ключевыми факторами, но поведенческий сегмент — третий по значимости признак модели, что подтверждает гипотезу о влиянии профиля клиента; поведенческая сегментация информативнее географической.

*Вывод:* итоговый ROC-AUC 0.771 превышает целевые 0.75, характер ошибок определён (доминируют пропущенные задержки), важность признаков проанализирована.

## [11. Выводы и рекомендации](#content)<a id="11"></a>

Что нужно зафиксировать:

* **Результаты моделирования.** Укажите итоговое значение ROC-AUC на тестовой выборке. Удалось ли вам достичь целевого показателя? Насколько эффективно итоговая модель справляется с выявлением задержек по сравнению с базовыми?
* **Эффективность сегментации.** Сформулируйте вывод о полезности кластеризации. Подтвердилась ли гипотеза о том, что профиль клиента и его географическое положение влияют на риск задержки? Какие именно кластеры — географические или поведенческие — оказались более информативными для модели? Добавьте в раздел визуализации, которые вы получили, работая над проектом.
* **Технический инсайт.** Если вы экспериментировали с разным количеством  кластеров как с гиперпараметром, то укажите, какое количество кластеров K оказалось оптимальным. Кратко поясните, почему слишком большое K может вредить качеству прогноза.
* **Бизнес-рекомендации.** Предложите, как CargaPronto может использовать вашу модель в реальных операциях.